# Лабораторная работа № 4

## Хеширование

Цель работы — реализовать хеш-таблицы с открытой адресацией и методом цепочек, добавить автоматическое изменение размера и применить хеширование в практических задачах.


## Задание 1. Хеш-таблица с квадратичным пробированием

При коллизии проверяются позиции по формуле `(hash + i²) % size`. При загрузке выше `0.7` таблица увеличивается, а при загрузке ниже `0.2` — уменьшается. Новый размер является ближайшим подходящим простым числом.


In [1]:
def is_prime(number):
    if number < 2:
        return False
    if number % 2 == 0:
        return number == 2
    divisor = 3
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 2
    return True


def next_prime(number):
    candidate = max(2, number)
    while not is_prime(candidate):
        candidate += 1
    return candidate


class QuadraticHashTable:
    _DELETED = object()
    _MISSING = object()

    def __init__(self, capacity=7):
        self._capacity = next_prime(capacity)
        self._min_capacity = self._capacity
        self._slots = [None] * self._capacity
        self._count = 0

    def __len__(self):
        return self._count

    @property
    def capacity(self):
        return self._capacity

    @property
    def load_factor(self):
        return self._count / self._capacity

    def _hash(self, key):
        if not isinstance(key, int):
            raise TypeError("в этой версии ключ должен быть целым числом")
        return key % self._capacity

    def _find_slot(self, key, for_insert=False):
        start = self._hash(key)
        first_deleted = None
        for i in range(self._capacity):
            index = (start + i * i) % self._capacity
            slot = self._slots[index]
            if slot is None:
                return first_deleted if for_insert and first_deleted is not None else index
            if slot is self._DELETED:
                if for_insert and first_deleted is None:
                    first_deleted = index
            elif slot[0] == key:
                return index
        return first_deleted if for_insert else None

    def _insert_without_resize(self, key, value):
        index = self._find_slot(key, for_insert=True)
        if index is None:
            return False
        self._slots[index] = (key, value)
        self._count += 1
        return True

    def _resize(self, new_capacity):
        items = list(self.items())
        self._capacity = next_prime(max(self._min_capacity, new_capacity))
        self._slots = [None] * self._capacity
        self._count = 0
        for key, value in items:
            if not self._insert_without_resize(key, value):
                raise RuntimeError("не удалось перестроить хеш-таблицу")

    def put(self, key, value):
        existing = self._find_slot(key)
        if existing is not None and self._slots[existing] not in (None, self._DELETED):
            self._slots[existing] = (key, value)
            return

        if (self._count + 1) / self._capacity > 0.7:
            self._resize(self._capacity * 2)

        while not self._insert_without_resize(key, value):
            self._resize(self._capacity * 2)

    def get(self, key, default=_MISSING):
        index = self._find_slot(key)
        if index is None or self._slots[index] in (None, self._DELETED):
            if default is self._MISSING:
                raise KeyError(key)
            return default
        return self._slots[index][1]

    def __setitem__(self, key, value):
        self.put(key, value)

    def __getitem__(self, key):
        return self.get(key)

    def __contains__(self, key):
        index = self._find_slot(key)
        return index is not None and self._slots[index] not in (None, self._DELETED)

    def __delitem__(self, key):
        index = self._find_slot(key)
        if index is None or self._slots[index] in (None, self._DELETED):
            raise KeyError(key)
        self._slots[index] = self._DELETED
        self._count -= 1

        if self._capacity > self._min_capacity and self.load_factor < 0.2:
            self._resize(max(self._min_capacity, self._capacity // 2))

    def items(self):
        for slot in self._slots:
            if slot is not None and slot is not self._DELETED:
                yield slot

    def __str__(self):
        return str(dict(self.items()))


In [2]:
table = QuadraticHashTable(7)

# Эти ключи имеют одинаковый начальный хеш при размере 7.
table[1] = "один"
table[8] = "восемь"
table[15] = "пятнадцать"
assert table[1] == "один" and table[8] == "восемь" and table[15] == "пятнадцать"

for number in range(20, 40):
    table[number] = number * 10
grown_capacity = table.capacity
assert grown_capacity > 7
assert len(table) == 23
assert 15 in table and 100 not in table

for number in range(20, 40):
    del table[number]
assert len(table) == 3
assert table.capacity < grown_capacity

del table[8]
assert 8 not in table and len(table) == 2

print("Содержимое:", table)
print("Размер таблицы:", table.capacity)
print("Количество элементов:", len(table))
print("Все тесты задания 1 пройдены.")


Содержимое: {1: 'один', 15: 'пятнадцать'}
Размер таблицы: 7
Количество элементов: 2
Все тесты задания 1 пройдены.


## Задание 2. Хеш-таблица методом цепочек

Каждый слот содержит односвязный список `UnorderedList`. Элементы с одинаковым хешем хранятся в одной цепочке. Правила увеличения и уменьшения размера совпадают с заданием 1.


In [3]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None


class UnorderedList:
    def __init__(self):
        self.head = None

    def add(self, item):
        node = Node(item)
        node.next = self.head
        self.head = node

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.data
            current = current.next

    def is_empty(self):
        return self.head is None


class ChainedHashTable:
    _MISSING = object()

    def __init__(self, capacity=7):
        self._capacity = next_prime(capacity)
        self._min_capacity = self._capacity
        self._buckets = [UnorderedList() for _ in range(self._capacity)]
        self._count = 0

    def __len__(self):
        return self._count

    @property
    def capacity(self):
        return self._capacity

    @property
    def load_factor(self):
        return self._count / self._capacity

    def _index(self, key):
        return hash(key) % self._capacity

    def _find_node(self, key):
        current = self._buckets[self._index(key)].head
        while current is not None:
            if current.data[0] == key:
                return current
            current = current.next
        return None

    def _put_without_resize(self, key, value):
        self._buckets[self._index(key)].add((key, value))
        self._count += 1

    def _resize(self, new_capacity):
        items = list(self.items())
        self._capacity = next_prime(max(self._min_capacity, new_capacity))
        self._buckets = [UnorderedList() for _ in range(self._capacity)]
        self._count = 0
        for key, value in items:
            self._put_without_resize(key, value)

    def put(self, key, value):
        node = self._find_node(key)
        if node is not None:
            node.data = (key, value)
            return

        if (self._count + 1) / self._capacity > 0.7:
            self._resize(self._capacity * 2)
        self._put_without_resize(key, value)

    def get(self, key, default=_MISSING):
        node = self._find_node(key)
        if node is None:
            if default is self._MISSING:
                raise KeyError(key)
            return default
        return node.data[1]

    def __setitem__(self, key, value):
        self.put(key, value)

    def __getitem__(self, key):
        return self.get(key)

    def __contains__(self, key):
        return self._find_node(key) is not None

    def __delitem__(self, key):
        bucket = self._buckets[self._index(key)]
        previous = None
        current = bucket.head
        while current is not None and current.data[0] != key:
            previous, current = current, current.next
        if current is None:
            raise KeyError(key)
        if previous is None:
            bucket.head = current.next
        else:
            previous.next = current.next
        self._count -= 1

        if self._capacity > self._min_capacity and self.load_factor < 0.2:
            self._resize(max(self._min_capacity, self._capacity // 2))

    def items(self):
        for bucket in self._buckets:
            yield from bucket

    def __str__(self):
        return str(dict(self.items()))


In [4]:
chains = ChainedHashTable(7)
chains[1] = "A"
chains[8] = "B"
chains[15] = "C"
assert chains[1] == "A" and chains[8] == "B" and chains[15] == "C"

for number in range(20):
    chains[number + 100] = number
grown_capacity = chains.capacity
assert grown_capacity > 7 and len(chains) == 23

for number in range(20):
    del chains[number + 100]
assert chains.capacity < grown_capacity
assert len(chains) == 3 and 8 in chains

del chains[8]
assert 8 not in chains

print("Содержимое:", chains)
print("Размер таблицы:", chains.capacity)
print("Все тесты задания 2 пройдены.")


Содержимое: {15: 'C', 1: 'A'}
Размер таблицы: 7
Все тесты задания 2 пройдены.


## Задание 3. Строковые ключи

Для строк используется простой полиномиальный хеш: символы последовательно добавляются к накопленному значению с множителем `31`.


In [5]:
class StringHashTable(QuadraticHashTable):
    def _hash(self, key):
        if not isinstance(key, str):
            raise TypeError("ключ должен быть строкой")
        value = 0
        for character in key:
            value = (value * 31 + ord(character)) % self._capacity
        return value


words = StringHashTable()
words["кот"] = 1
words["дом"] = 2
words["Повторяю"] = 3
words["кот"] = 10

assert words["кот"] == 10
assert words["дом"] == 2
assert "Повторяю" in words and "повторяю" not in words
assert len(words) == 3

del words["дом"]
assert "дом" not in words

print("Таблица со строковыми ключами:", words)
print("Тесты задания 3 пройдены.")


Таблица со строковыми ключами: {'Повторяю': 3, 'кот': 10}
Тесты задания 3 пройдены.


## Задание 4. Номер вхождения каждого слова

Регистр сохраняется: слова `Раз` и `раз` считаются разными. Для каждого слова в хеш-таблице хранится число уже найденных вхождений.


In [6]:
import re


def word_occurrence_numbers(text):
    table = StringHashTable()
    result = []
    for word in re.findall(r"[А-Яа-яЁё]+", text):
        count = table.get(word, 0) + 1
        table[word] = count
        result.append(count)
    return result


example = "Раз раз раз как меня слышно Повторяю раз раз раз Повторяю"
result = word_occurrence_numbers(example)
expected = [1, 1, 2, 1, 1, 1, 1, 3, 4, 5, 2]
assert result == expected

print("Текст:", example)
print("Результат:", *result)


Текст: Раз раз раз как меня слышно Повторяю раз раз раз Повторяю
Результат: 1 1 2 1 1 1 1 3 4 5 2


## Задание 5. Регистрация и авторизация

Для каждого пользователя сохраняются логин, случайная соль длиной 32 байта и хеш пароля длиной 32 байта. Пароль хешируется функцией `scrypt`, а сравнение выполняется через `secrets.compare_digest`.


In [7]:
import hashlib
import json
import secrets
from pathlib import Path
from tempfile import TemporaryDirectory


class AuthSystem:
    def __init__(self, database_path):
        self.database_path = Path(database_path)
        if not self.database_path.exists():
            self.database_path.write_text("{}", encoding="utf-8")

    def _load(self):
        return json.loads(self.database_path.read_text(encoding="utf-8"))

    def _save(self, users):
        self.database_path.write_text(
            json.dumps(users, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    @staticmethod
    def _password_hash(password, salt):
        return hashlib.scrypt(
            password.encode("utf-8"),
            salt=salt,
            n=2**14,
            r=8,
            p=1,
            dklen=32,
        )

    def register(self, login, password):
        users = self._load()
        if login in users:
            raise ValueError("пользователь уже существует")
        salt = secrets.token_bytes(32)
        password_hash = self._password_hash(password, salt)
        users[login] = {
            "salt": salt.hex(),
            "password_hash": password_hash.hex(),
        }
        self._save(users)

    def authenticate(self, login, password):
        users = self._load()
        if login not in users:
            return False
        salt = bytes.fromhex(users[login]["salt"])
        saved_hash = bytes.fromhex(users[login]["password_hash"])
        entered_hash = self._password_hash(password, salt)
        return secrets.compare_digest(entered_hash, saved_hash)


In [8]:
with TemporaryDirectory() as directory:
    auth = AuthSystem(Path(directory) / "users.json")
    auth.register("anton", "StrongPassword123")

    assert auth.authenticate("anton", "StrongPassword123")
    assert not auth.authenticate("anton", "wrong-password")
    assert not auth.authenticate("unknown", "StrongPassword123")

    stored = json.loads((Path(directory) / "users.json").read_text(encoding="utf-8"))
    assert stored["anton"]["password_hash"] != "StrongPassword123"
    assert len(bytes.fromhex(stored["anton"]["salt"])) == 32
    assert len(bytes.fromhex(stored["anton"]["password_hash"])) == 32

print("Регистрация выполнена.")
print("Правильный пароль принят, неправильный отклонён.")
print("Тесты задания 5 пройдены.")


Регистрация выполнена.
Правильный пароль принят, неправильный отклонён.
Тесты задания 5 пройдены.


## Задание 6. Поиск дубликатов файлов

Для каждого файла непосредственно в указанной директории вычисляется SHA-256. Файлы с одинаковым хешем объединяются в группы дубликатов.


In [9]:
def file_hash(path, chunk_size=65_536):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def find_duplicates(directory):
    directory = Path(directory)
    if not directory.is_dir():
        raise ValueError("указанный путь не является директорией")

    files_by_hash = {}
    for path in sorted(directory.iterdir()):
        if path.is_file():
            digest = file_hash(path)
            files_by_hash.setdefault(digest, []).append(path)

    return [paths for paths in files_by_hash.values() if len(paths) > 1]


def find_duplicates_from_input(input_func=input):
    return find_duplicates(input_func("Введите путь к директории: "))


with TemporaryDirectory() as directory:
    directory = Path(directory)
    (directory / "copy-1.txt").write_text("одинаковые данные", encoding="utf-8")
    (directory / "copy-2.txt").write_text("одинаковые данные", encoding="utf-8")
    (directory / "unique.txt").write_text("другие данные", encoding="utf-8")

    duplicates = find_duplicates(directory)
    duplicate_names = [[path.name for path in group] for group in duplicates]
    assert duplicate_names == [["copy-1.txt", "copy-2.txt"]]

print("Найдены дубликаты:", duplicate_names)
print("Тесты задания 6 пройдены.")


Найдены дубликаты: [['copy-1.txt', 'copy-2.txt']]
Тесты задания 6 пройдены.


## Вывод

В работе реализованы две хеш-таблицы: с квадратичным пробированием и методом цепочек. Для обеих таблиц добавлены операции поиска, вставки, удаления и автоматического изменения размера. Хеширование применено для подсчёта слов, безопасного хранения паролей и поиска одинаковых файлов.
